In [1]:
# Cell 1: Setup
import sys, os, torch
import numpy as np
import matplotlib.pyplot as plt
sys.path.append(os.path.abspath(os.path.join('..')))
from load_dataset import load_mitbih_dataset
from run import *
from models import DeepECG, ResNet1D
import warnings
warnings.filterwarnings("ignore")
from visualizations import Visualizer 

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Framework loaded. Running on {device}")

# Set random seed
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)

Framework loaded. Running on cpu


In [2]:
print("\n=== TASKS 1-3: Random Split ===")
print("Loading Full MIT-BIH records (0-100%) for Random Split...")

# Load full records (range 0.0 to 1.0)
loader = load_mitbih_dataset(
    num_beats=3, 
    enrollment_range=(0.0, 1.0), # Load everything
    cleanup_zip=False
)
# We use load_session("Session_1") effectively to get the whole file here
x_all, y_all = loader.load_session("Session_1") 

print(f"Data Loaded: {x_all.shape} samples, {len(np.unique(y_all))} subjects")

# Run Standard Benchmarks
print("\n[Task 1] Closed-Set Identification")
run_closed_set_identification(x_all, y_all, DeepECG, epochs=2, device=device)

print("\n[Task 2] Verification")
run_verification(x_all, y_all, DeepECG, epochs=2, device=device)

print("\n[Task 3] Subject-Disjoint Verification")
run_subject_disjoint_verification(x_all, y_all, DeepECG, epochs=2, device=device)


=== TASKS 1-3: Random Split ===
Loading Full MIT-BIH records (0-100%) for Random Split...


Processing Session_1: 100%|██████████| 69/69 [11:46<00:00, 10.25s/it]


Data Loaded: (121327, 216) samples, 69 subjects

[Task 1] Closed-Set Identification

[TASK] Closed-Set Identification on cpu
    Epoch 001 | Loss: 1.5438
    Epoch 002 | Loss: 0.8149
[RESULT] Closed-Set Accuracy: 0.7535

[Task 2] Verification

[TASK] Verification (Random Split / Closed-Set) on cpu
[INFO] Phase 1: Training feature extractor (Learning Identity)...
    Epoch 001 | Loss: 1.6289
    Epoch 002 | Loss: 0.8533
[INFO] Phase 2: Computing EER using 'balanced' sampling...
[INFO] Generating 10000 BALANCED pairs (50/50 split)...
[RESULT] Mode: BALANCED | EER: 0.1060 | AUC: 0.9630

[Task 3] Subject-Disjoint Verification

[TASK] Subject-Disjoint Verification (Open-Set) on cpu
[INFO] Splitting: 48 Training Subjects vs 21 Test Subjects
[INFO] Phase 1: Training on Known Subjects...
    Epoch 001 | Loss: 1.3783
    Epoch 002 | Loss: 0.6726
[INFO] Phase 2: Computing EER on 21 Unseen Subjects...
[INFO] Generating 10000 BALANCED pairs (50/50 split)...
[RESULT] Mode: BALANCED | EER: 0.1242 | 

{'eer': 0.124200000000356, 'auc': 0.9440168000000001}

In [3]:
print("\n=== TASK 4: Biometric Regimes (Table 1) ===")

# A. SHORT-TERM STABILITY (Half vs Half)
# Split: First 15 min (0.0-0.5) vs Last 15 min (0.5-1.0)
print("\n[A] Loading Short-Term Regime (First 15m vs Last 15m)...")
loader_st = load_mitbih_dataset(
    num_beats=3, 
    enrollment_range=(0.0, 0.5), 
    probe_range=(0.5, 1.0),
    cleanup_zip=False
)
x_st_enr, y_st_enr = loader_st.load_session("Session_1")
x_st_prb, y_st_prb = loader_st.load_session("Session_2")
print(f"    Short-Term Shapes: Enroll {x_st_enr.shape}, Probe {x_st_prb.shape}")

print("    Running Verification...")
run_cross_session_verification(x_st_enr, y_st_enr, x_st_prb, y_st_prb, DeepECG, epochs=2, device=device, visualize=False)


# B. MAXIMAL INTERVAL (Long-Term Proxy)
# Split: First 2 min vs Last 2 min
# 2 min / 30 min = ~0.067
print("\n[B] Loading Maximal Regime (First 2m vs Last 2m)...")
loader_max = load_mitbih_dataset(
    num_beats=3, 
    enrollment_range=(0.0, 0.067), # First ~2 mins
    probe_range=(0.933, 1.0),      # Last ~2 mins
    cleanup_zip=False
)
x_mx_enr, y_mx_enr = loader_max.load_session("Session_1")
x_mx_prb, y_mx_prb = loader_max.load_session("Session_2")
print(f"    Maximal Shapes: Enroll {x_mx_enr.shape}, Probe {x_mx_prb.shape}")

print("    Running Verification...")
run_cross_session_verification(x_mx_enr, y_mx_enr, x_mx_prb, y_mx_prb, DeepECG, epochs=2, device=device, visualize=False)


=== TASK 4: Biometric Regimes (Table 1) ===

[A] Loading Short-Term Regime (First 15m vs Last 15m)...


Processing Session_2: 100%|██████████| 69/69 [10:53<00:00,  9.47s/it]


    Short-Term Shapes: Enroll (60648, 216), Probe (60679, 216)
    Running Verification...

[TASK] Cross-Session Verification (EER) on cpu
[INFO] Phase 1: Training Feature Extractor on Session 1...
    Epoch 001 | Loss: 1.8296
    Epoch 002 | Loss: 0.9131
[INFO] Phase 2: Extracting Embeddings for Pairing...
[INFO] Generating 10000 Cross-Session Pairs (balanced)...
[RESULT] Cross-Session EER: 0.1054 | AUC: 0.9567

[B] Loading Maximal Regime (First 2m vs Last 2m)...


Processing Session_2: 100%|██████████| 69/69 [16:02<00:00, 13.95s/it]


    Maximal Shapes: Enroll (8097, 216), Probe (8166, 216)
    Running Verification...

[TASK] Cross-Session Verification (EER) on cpu
[INFO] Phase 1: Training Feature Extractor on Session 1...
    Epoch 001 | Loss: 3.5849
    Epoch 002 | Loss: 2.1973
[INFO] Phase 2: Extracting Embeddings for Pairing...
[INFO] Generating 10000 Cross-Session Pairs (balanced)...
[RESULT] Cross-Session EER: 0.1812 | AUC: 0.8991


{'eer': 0.18119999999945763, 'auc': 0.8990950799999999}

In [4]:
print("\n=== TASK 5: Blind Segmentation (Short-Term Regime) ===")

blind_params = {
    'mode': 'blind',
    'window_len': 5.0,  
    'stride': 2.0,      
    'bandpass': True,
    'normalize': 'zscore'
}

# Use Short-Term Split (50/50) for Blind Test
loader_blind = load_mitbih_dataset(
    num_beats=1, 
    enrollment_range=(0.0, 0.5),
    probe_range=(0.5, 1.0),
    preprocessing_params=blind_params, 
    cleanup_zip=False
)

x_blind_enr, y_blind_enr = loader_blind.load_session("Session_1")
x_blind_prb, y_blind_prb = loader_blind.load_session("Session_2")
print(f"Blind Data Shapes: Enroll {x_blind_enr.shape}, Probe {x_blind_prb.shape}")

print("\n[Blind] Identification")
run_cross_session_identification(x_blind_enr, y_blind_enr, x_blind_prb, y_blind_prb, DeepECG, epochs=2, device=device)

print("\n[Blind] Verification")
run_cross_session_verification(x_blind_enr, y_blind_enr, x_blind_prb, y_blind_prb, DeepECG, epochs=2, device=device, visualize=False)


=== TASK 5: Blind Segmentation (Short-Term Regime) ===


Processing Session_2: 100%|██████████| 69/69 [00:02<00:00, 31.14it/s]


Blind Data Shapes: Enroll (24127, 1800), Probe (24173, 1800)

[Blind] Identification

[TASK] Cross-Session Identification (Rank-1) on cpu
[INFO] Subjects: 69 in Train, 69 in Test.
[INFO] Evaluated on intersection: 69 common subjects.
[INFO] Phase 1: Training Classifier on Session 1...
    Epoch 001 | Loss: 2.3860
    Epoch 002 | Loss: 0.9230
[INFO] Phase 2: Predicting on Session 2...
[RESULT] Cross-Session Identification Accuracy: 68.94%

[Blind] Verification

[TASK] Cross-Session Verification (EER) on cpu
[INFO] Phase 1: Training Feature Extractor on Session 1...
    Epoch 001 | Loss: 2.3860
    Epoch 002 | Loss: 0.9230
[INFO] Phase 2: Extracting Embeddings for Pairing...
[INFO] Generating 10000 Cross-Session Pairs (balanced)...
[RESULT] Cross-Session EER: 0.0856 | AUC: 0.9691


{'eer': 0.0855999999999886, 'auc': 0.96909652}